<a href="https://colab.research.google.com/github/AshokGit544/Autonomous_AI_Data_Platform/blob/main/Autonomous_AI_Data_Platform.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
!pip -q install pandas numpy scikit-learn faiss-cpu pyspark

In [14]:
import random
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import pandas as pd

random.seed(42)
np.random.seed(42)

PROJECT_DIR = Path("Autonomous-AI-Data-Platform")
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
OUTPUT_DIR = PROJECT_DIR / "data" / "output"
VECTOR_DIR = PROJECT_DIR / "vector_store"
CONFIG_DIR = PROJECT_DIR / "config"

for folder in [RAW_DIR, PROCESSED_DIR, OUTPUT_DIR, VECTOR_DIR, CONFIG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project structure created successfully")
print("Base path:", PROJECT_DIR.resolve())

Project structure created successfully
Base path: /content/Autonomous-AI-Data-Platform


In [15]:
num_meters = 500
num_assets = 120
num_days = 30

meter_ids = [f"MTR{i:04d}" for i in range(1, num_meters + 1)]
asset_ids = [f"AST{i:04d}" for i in range(1, num_assets + 1)]
regions = ["North", "South", "East", "West", "Central"]
asset_types = ["Transformer", "Feeder", "Breaker", "Line", "Substation"]
weather_conditions = ["Normal", "Storm", "Heatwave", "Coldwave", "Windy"]

asset_master = pd.DataFrame({
    "asset_id": asset_ids,
    "asset_type": np.random.choice(asset_types, num_assets),
    "region": np.random.choice(regions, num_assets),
    "installation_year": np.random.randint(1998, 2023, num_assets),
    "health_score": np.random.randint(55, 100, num_assets)
})

meter_asset_map = pd.DataFrame({
    "meter_id": meter_ids,
    "asset_id": np.random.choice(asset_ids, num_meters)
}).merge(asset_master[["asset_id", "region"]], on="asset_id", how="left")

dates = [datetime(2025, 1, 1) + timedelta(days=i) for i in range(num_days)]

meter_rows = []
weather_rows = []
outage_rows = []

for d in dates:
    for region in regions:
        base_temp = np.random.randint(15, 35)
        condition = np.random.choice(weather_conditions, p=[0.45, 0.15, 0.15, 0.1, 0.15])
        wind_speed = np.random.randint(5, 45)
        weather_rows.append([
            d.date().isoformat(),
            region,
            base_temp,
            wind_speed,
            condition
        ])

for meter_id in meter_ids:
    meter_region = meter_asset_map.loc[meter_asset_map["meter_id"] == meter_id, "region"].values[0]
    linked_asset = meter_asset_map.loc[meter_asset_map["meter_id"] == meter_id, "asset_id"].values[0]

    for d in dates:
        weather_row = [r for r in weather_rows if r[0] == d.date().isoformat() and r[1] == meter_region][0]
        temp = weather_row[2]
        wind = weather_row[3]
        condition = weather_row[4]

        base_usage = np.random.normal(75, 15)
        base_voltage = np.random.normal(120, 3)

        if condition == "Heatwave":
            base_usage += np.random.uniform(20, 45)
        if condition == "Storm":
            base_voltage -= np.random.uniform(4, 12)
        if condition == "Coldwave":
            base_usage += np.random.uniform(10, 20)
        if wind > 30:
            base_voltage -= np.random.uniform(2, 6)

        if np.random.rand() < 0.015:
            base_usage *= np.random.uniform(1.8, 2.8)

        if np.random.rand() < 0.012:
            base_voltage -= np.random.uniform(10, 18)

        meter_rows.append([
            meter_id,
            linked_asset,
            meter_region,
            d.date().isoformat(),
            round(base_usage, 2),
            round(base_voltage, 2),
            condition
        ])

for i in range(1, 81):
    outage_date = random.choice(dates).date().isoformat()
    asset = asset_master.sample(1).iloc[0]
    probable_cause = np.random.choice(
        ["vegetation contact", "transformer overload", "weather damage", "equipment aging", "breaker trip"],
        p=[0.2, 0.25, 0.25, 0.2, 0.1]
    )
    outage_rows.append([
        f"OUT{i:04d}",
        asset["asset_id"],
        asset["region"],
        outage_date,
        probable_cause,
        np.random.randint(20, 240)
    ])

meter_readings = pd.DataFrame(meter_rows, columns=[
    "meter_id", "asset_id", "region", "reading_date", "usage_kwh", "voltage", "weather_condition"
])

weather_data = pd.DataFrame(weather_rows, columns=[
    "weather_date", "region", "temperature_c", "wind_speed", "weather_condition"
])

outage_history = pd.DataFrame(outage_rows, columns=[
    "outage_id", "asset_id", "region", "outage_date", "probable_cause", "duration_minutes"
])

feedback_log = pd.DataFrame([
    ["high_usage_spike", 1.60, "active"],
    ["low_voltage_drop", 108.00, "active"],
    ["retrieval_min_score", 0.72, "active"]
], columns=["rule_name", "rule_value", "status"])

asset_master.to_csv(RAW_DIR / "asset_master.csv", index=False)
meter_asset_map.to_csv(RAW_DIR / "meter_asset_map.csv", index=False)
meter_readings.to_csv(RAW_DIR / "meter_readings.csv", index=False)
weather_data.to_csv(RAW_DIR / "weather_data.csv", index=False)
outage_history.to_csv(RAW_DIR / "outage_history.csv", index=False)
feedback_log.to_csv(CONFIG_DIR / "feedback_rules.csv", index=False)

print("Synthetic utility data created successfully")
print("asset_master:", asset_master.shape)
print("meter_asset_map:", meter_asset_map.shape)
print("meter_readings:", meter_readings.shape)
print("weather_data:", weather_data.shape)
print("outage_history:", outage_history.shape)
print("feedback_rules:", feedback_log.shape)

Synthetic utility data created successfully
asset_master: (120, 5)
meter_asset_map: (500, 3)
meter_readings: (15000, 7)
weather_data: (150, 5)
outage_history: (80, 6)
feedback_rules: (3, 3)


In [16]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, stddev, when, lit, concat_ws, round as spark_round

spark = SparkSession.builder.appName("AutonomousAIDataPlatform").getOrCreate()

asset_spark = spark.read.option("header", True).csv(str(RAW_DIR / "asset_master.csv"), inferSchema=True)
meter_map_spark = spark.read.option("header", True).csv(str(RAW_DIR / "meter_asset_map.csv"), inferSchema=True)
meter_spark = spark.read.option("header", True).csv(str(RAW_DIR / "meter_readings.csv"), inferSchema=True)
weather_spark = spark.read.option("header", True).csv(str(RAW_DIR / "weather_data.csv"), inferSchema=True)
outage_spark = spark.read.option("header", True).csv(str(RAW_DIR / "outage_history.csv"), inferSchema=True)
rules_spark = spark.read.option("header", True).csv(str(CONFIG_DIR / "feedback_rules.csv"), inferSchema=True)

meter_enriched = (
    meter_spark
    .join(asset_spark, on=["asset_id", "region"], how="left")
    .join(
        weather_spark,
        (meter_spark["reading_date"] == weather_spark["weather_date"]) &
        (meter_spark["region"] == weather_spark["region"]),
        how="left"
    )
    .select(
        meter_spark["meter_id"],
        meter_spark["asset_id"],
        meter_spark["region"],
        meter_spark["reading_date"],
        meter_spark["usage_kwh"],
        meter_spark["voltage"],
        meter_spark["weather_condition"].alias("meter_weather_condition"),
        asset_spark["asset_type"],
        asset_spark["installation_year"],
        asset_spark["health_score"],
        weather_spark["temperature_c"],
        weather_spark["wind_speed"],
        weather_spark["weather_condition"].alias("regional_weather_condition")
    )
)

baseline_stats = meter_enriched.groupBy("meter_id").agg(
    avg("usage_kwh").alias("avg_usage"),
    stddev("usage_kwh").alias("std_usage"),
    avg("voltage").alias("avg_voltage"),
    stddev("voltage").alias("std_voltage")
)

processed = (
    meter_enriched
    .join(baseline_stats, on="meter_id", how="left")
    .withColumn(
        "usage_ratio",
        spark_round(col("usage_kwh") / col("avg_usage"), 2)
    )
    .withColumn(
        "voltage_drop",
        spark_round(col("avg_voltage") - col("voltage"), 2)
    )
    .withColumn(
        "usage_anomaly_flag",
        when(col("usage_kwh") > col("avg_usage") * lit(1.60), lit(1)).otherwise(lit(0))
    )
    .withColumn(
        "voltage_anomaly_flag",
        when(col("voltage") < lit(108.0), lit(1)).otherwise(lit(0))
    )
    .withColumn(
        "combined_anomaly_flag",
        when((col("usage_anomaly_flag") == 1) | (col("voltage_anomaly_flag") == 1), lit(1)).otherwise(lit(0))
    )
    .withColumn(
        "risk_hint",
        when((col("regional_weather_condition") == "Storm") & (col("voltage_anomaly_flag") == 1), lit("weather_risk"))
        .when((col("health_score") < 65) & (col("voltage_anomaly_flag") == 1), lit("asset_health_risk"))
        .when((col("usage_anomaly_flag") == 1) & (col("temperature_c") > 30), lit("heat_load_risk"))
        .otherwise(lit("normal_pattern"))
    )
)

outage_summary = outage_spark.groupBy("asset_id").count().withColumnRenamed("count", "historical_outage_count")

final_features = (
    processed
    .join(outage_summary, on="asset_id", how="left")
    .fillna({"historical_outage_count": 0})
    .withColumn(
        "retrieval_text",
        concat_ws(
            " ",
            lit("Meter"),
            col("meter_id"),
            lit("Asset"),
            col("asset_id"),
            lit("Region"),
            col("region"),
            lit("AssetType"),
            col("asset_type"),
            lit("HealthScore"),
            col("health_score"),
            lit("Usage"),
            col("usage_kwh"),
            lit("Voltage"),
            col("voltage"),
            lit("Weather"),
            col("regional_weather_condition"),
            lit("RiskHint"),
            col("risk_hint"),
            lit("HistoricalOutages"),
            col("historical_outage_count")
        )
    )
)

final_features.toPandas().to_csv(PROCESSED_DIR / "utility_ai_features.csv", index=False)

print("PySpark feature pipeline completed successfully")
print("Processed rows:", final_features.count())
final_features.show(5, truncate=False)

PySpark feature pipeline completed successfully
Processed rows: 15000
+--------+--------+------+------------+---------+-------+-----------------------+----------+-----------------+------------+-------------+----------+--------------------------+-----------------+------------------+------------------+------------------+-----------+------------+------------------+--------------------+---------------------+--------------+-----------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------+
|asset_id|meter_id|region|reading_date|usage_kwh|voltage|meter_weather_condition|asset_type|installation_year|health_score|temperature_c|wind_speed|regional_weather_condition|avg_usage        |std_usage         |avg_voltage       |std_voltage       |usage_ratio|voltage_drop|usage_anomaly_flag|voltage_anomaly_flag|combined_anomaly_flag|risk_hint     |historical_outage_count|retrieval_text   

In [17]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

features_df = pd.read_csv(PROCESSED_DIR / "utility_ai_features.csv")
rules_df = pd.read_csv(CONFIG_DIR / "feedback_rules.csv")

usage_threshold = float(rules_df.loc[rules_df["rule_name"] == "high_usage_spike", "rule_value"].iloc[0])
voltage_threshold = float(rules_df.loc[rules_df["rule_name"] == "low_voltage_drop", "rule_value"].iloc[0])

features_df["usage_ratio"] = pd.to_numeric(features_df["usage_ratio"], errors="coerce")
features_df["voltage"] = pd.to_numeric(features_df["voltage"], errors="coerce")
features_df["health_score"] = pd.to_numeric(features_df["health_score"], errors="coerce")
features_df["temperature_c"] = pd.to_numeric(features_df["temperature_c"], errors="coerce")
features_df["wind_speed"] = pd.to_numeric(features_df["wind_speed"], errors="coerce")
features_df["historical_outage_count"] = pd.to_numeric(features_df["historical_outage_count"], errors="coerce")
features_df["usage_anomaly_flag"] = pd.to_numeric(features_df["usage_anomaly_flag"], errors="coerce")
features_df["voltage_anomaly_flag"] = pd.to_numeric(features_df["voltage_anomaly_flag"], errors="coerce")

model_input = features_df[
    ["usage_ratio", "voltage", "health_score", "temperature_c", "wind_speed", "historical_outage_count"]
].fillna(0)

scaler = StandardScaler()
scaled_input = scaler.fit_transform(model_input)

iso = IsolationForest(n_estimators=150, contamination=0.03, random_state=42)
features_df["ml_anomaly_flag"] = (iso.fit_predict(scaled_input) == -1).astype(int)
features_df["final_anomaly_flag"] = (
    (features_df["usage_ratio"] > usage_threshold) |
    (features_df["voltage"] < voltage_threshold) |
    (features_df["ml_anomaly_flag"] == 1)
).astype(int)

features_df["feedback_status"] = np.where(
    (features_df["final_anomaly_flag"] == 1) &
    (features_df["risk_hint"] == "normal_pattern"),
    "review_threshold",
    "accepted"
)

review_rate = round((features_df["feedback_status"] == "review_threshold").mean(), 4)

updated_rules = rules_df.copy()

if review_rate > 0.08:
    updated_rules.loc[updated_rules["rule_name"] == "high_usage_spike", "rule_value"] = round(usage_threshold + 0.05, 2)

if features_df["voltage_anomaly_flag"].mean() < 0.01:
    updated_rules.loc[updated_rules["rule_name"] == "low_voltage_drop", "rule_value"] = round(voltage_threshold - 1.0, 2)

features_df.to_csv(OUTPUT_DIR / "anomaly_scored_output.csv", index=False)
updated_rules.to_csv(OUTPUT_DIR / "updated_feedback_rules.csv", index=False)

feedback_summary = pd.DataFrame({
    "metric": ["total_records", "ml_anomalies", "final_anomalies", "review_threshold_rate"],
    "value": [
        len(features_df),
        int(features_df["ml_anomaly_flag"].sum()),
        int(features_df["final_anomaly_flag"].sum()),
        review_rate
    ]
})
feedback_summary.to_csv(OUTPUT_DIR / "feedback_summary.csv", index=False)

print("Anomaly scoring and feedback update completed successfully")
print("Review threshold rate:", review_rate)
print("\nUpdated rules:")
print(updated_rules)
print("\nScored output preview:")
print(features_df[[
    "meter_id", "asset_id", "reading_date", "usage_ratio", "voltage",
    "risk_hint", "ml_anomaly_flag", "final_anomaly_flag", "feedback_status"
]].head(10))

Anomaly scoring and feedback update completed successfully
Review threshold rate: 0.0371

Updated rules:
             rule_name  rule_value  status
0     high_usage_spike        1.60  active
1     low_voltage_drop      108.00  active
2  retrieval_min_score        0.72  active

Scored output preview:
  meter_id asset_id reading_date  usage_ratio  voltage       risk_hint  \
0  MTR0352  AST0094   2025-01-01         0.81   109.46  normal_pattern   
1  MTR0352  AST0094   2025-01-02         1.03   112.33  normal_pattern   
2  MTR0352  AST0094   2025-01-03         0.80   118.65  normal_pattern   
3  MTR0352  AST0094   2025-01-04         1.06   124.16  normal_pattern   
4  MTR0352  AST0094   2025-01-05         0.85   120.81  normal_pattern   
5  MTR0352  AST0094   2025-01-06         1.01   116.96  normal_pattern   
6  MTR0352  AST0094   2025-01-07         1.11   101.37    weather_risk   
7  MTR0352  AST0094   2025-01-08         1.09   115.45  normal_pattern   
8  MTR0352  AST0094   2025-01-09 

In [18]:
import faiss
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
import pickle

scored_df = pd.read_csv(OUTPUT_DIR / "anomaly_scored_output.csv")
outage_history_df = pd.read_csv(RAW_DIR / "outage_history.csv")
asset_master_df = pd.read_csv(RAW_DIR / "asset_master.csv")

historical_docs = outage_history_df.merge(asset_master_df, on=["asset_id", "region"], how="left")
historical_docs["incident_text"] = (
    "Outage " + historical_docs["outage_id"].astype(str) +
    " Asset " + historical_docs["asset_id"].astype(str) +
    " Region " + historical_docs["region"].astype(str) +
    " Cause " + historical_docs["probable_cause"].astype(str) +
    " Duration " + historical_docs["duration_minutes"].astype(str) +
    " AssetType " + historical_docs["asset_type"].astype(str) +
    " HealthScore " + historical_docs["health_score"].astype(str)
)

retrieval_corpus = historical_docs["incident_text"].tolist()
feature_corpus = scored_df["retrieval_text"].fillna("").tolist()

combined_corpus = retrieval_corpus + feature_corpus

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(combined_corpus).astype(np.float32)
X = normalize(X, norm="l2", axis=1)

dense_vectors = X.toarray().astype("float32")
index = faiss.IndexFlatIP(dense_vectors.shape[1])
index.add(dense_vectors)

with open(VECTOR_DIR / "tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

faiss.write_index(index, str(VECTOR_DIR / "utility_incident_faiss.index"))

metadata = pd.DataFrame({
    "doc_id": list(range(len(combined_corpus))),
    "doc_type": ["historical_incident"] * len(retrieval_corpus) + ["live_meter_event"] * len(feature_corpus),
    "content": combined_corpus
})
metadata.to_csv(VECTOR_DIR / "vector_metadata.csv", index=False)

query_text = scored_df.loc[scored_df["final_anomaly_flag"] == 1, "retrieval_text"].iloc[0]
query_vector = vectorizer.transform([query_text]).astype(np.float32)
query_vector = normalize(query_vector, norm="l2", axis=1).toarray().astype("float32")

scores, indices = index.search(query_vector, 5)

retrieved_results = metadata.iloc[indices[0]].copy()
retrieved_results["score"] = scores[0]
retrieved_results.to_csv(OUTPUT_DIR / "top_similar_incidents.csv", index=False)

print("FAISS retrieval layer created successfully")
print("Sample anomaly query:")
print(query_text[:300])
print("\nTop similar retrieved results:")
print(retrieved_results[["doc_type", "score", "content"]])

FAISS retrieval layer created successfully
Sample anomaly query:
Meter MTR0352 Asset AST0094 Region South AssetType Breaker HealthScore 88 Usage 90.9 Voltage 101.37 Weather Storm RiskHint weather_risk HistoricalOutages 0

Top similar retrieved results:
             doc_type     score  \
86   live_meter_event  1.000000   
104  live_meter_event  0.697903   
103  live_meter_event  0.695632   
101  live_meter_event  0.668203   
97   live_meter_event  0.654737   

                                               content  
86   Meter MTR0352 Asset AST0094 Region South Asset...  
104  Meter MTR0352 Asset AST0094 Region South Asset...  
103  Meter MTR0352 Asset AST0094 Region South Asset...  
101  Meter MTR0352 Asset AST0094 Region South Asset...  
97   Meter MTR0352 Asset AST0094 Region South Asset...  


In [19]:
scored_df = pd.read_csv(OUTPUT_DIR / "anomaly_scored_output.csv")
retrieved_results = pd.read_csv(OUTPUT_DIR / "top_similar_incidents.csv")

anomalies = scored_df[scored_df["final_anomaly_flag"] == 1].copy()

def generate_root_cause_explanation(row, retrieved_df):
    similar_text = " | ".join(retrieved_df["content"].head(2).tolist())

    if row["risk_hint"] == "weather_risk":
        cause = "Likely weather-driven voltage instability"
        action = "Inspect weather-affected assets and verify field conditions"
    elif row["risk_hint"] == "asset_health_risk":
        cause = "Likely asset deterioration or overloaded equipment"
        action = "Prioritize maintenance review for low-health assets"
    elif row["risk_hint"] == "heat_load_risk":
        cause = "Likely demand spike caused by high temperature conditions"
        action = "Review peak-load behavior and transformer loading"
    elif row["ml_anomaly_flag"] == 1:
        cause = "Unusual behavior detected by anomaly model"
        action = "Send record for engineering review and threshold analysis"
    else:
        cause = "Pattern needs further investigation"
        action = "Review meter history and related asset behavior"

    confidence = "high" if row["risk_hint"] != "normal_pattern" else "medium"

    explanation = (
        f"Meter {row['meter_id']} linked to asset {row['asset_id']} in region {row['region']} "
        f"shows abnormal behavior on {row['reading_date']}. "
        f"The platform observed usage ratio {row['usage_ratio']} and voltage {row['voltage']}. "
        f"Primary risk interpretation is: {cause}. "
        f"Recommended action: {action}. "
        f"Similar historical signals were retrieved from past incidents: {similar_text}. "
        f"Confidence level: {confidence}."
    )
    return explanation, cause, action, confidence

explanations = []
for _, row in anomalies.head(25).iterrows():
    explanation, cause, action, confidence = generate_root_cause_explanation(row, retrieved_results)
    explanations.append([
        row["meter_id"],
        row["asset_id"],
        row["reading_date"],
        cause,
        action,
        confidence,
        explanation
    ])

ai_explanations_df = pd.DataFrame(explanations, columns=[
    "meter_id",
    "asset_id",
    "reading_date",
    "predicted_root_cause",
    "recommended_action",
    "confidence_level",
    "ai_explanation"
])

ai_explanations_df.to_csv(OUTPUT_DIR / "ai_root_cause_explanations.csv", index=False)

print("AI explanation layer completed successfully")
print(ai_explanations_df.head(10))

AI explanation layer completed successfully
  meter_id asset_id reading_date                        predicted_root_cause  \
0  MTR0352  AST0094   2025-01-07   Likely weather-driven voltage instability   
1  MTR0401  AST0006   2025-01-01  Unusual behavior detected by anomaly model   
2  MTR0401  AST0006   2025-01-02  Unusual behavior detected by anomaly model   
3  MTR0401  AST0006   2025-01-03  Unusual behavior detected by anomaly model   
4  MTR0401  AST0006   2025-01-04  Unusual behavior detected by anomaly model   
5  MTR0401  AST0006   2025-01-05  Unusual behavior detected by anomaly model   
6  MTR0401  AST0006   2025-01-06  Unusual behavior detected by anomaly model   
7  MTR0401  AST0006   2025-01-07  Unusual behavior detected by anomaly model   
8  MTR0401  AST0006   2025-01-09  Unusual behavior detected by anomaly model   
9  MTR0401  AST0006   2025-01-10  Unusual behavior detected by anomaly model   

                                  recommended_action confidence_level  \
0 

In [20]:
ai_explanations_df = pd.read_csv(OUTPUT_DIR / "ai_root_cause_explanations.csv")
updated_rules_df = pd.read_csv(OUTPUT_DIR / "updated_feedback_rules.csv")
retrieved_results_df = pd.read_csv(OUTPUT_DIR / "top_similar_incidents.csv")

def score_explanation(row):
    score = 0.0

    if isinstance(row["predicted_root_cause"], str) and len(row["predicted_root_cause"].strip()) > 8:
        score += 0.25
    if isinstance(row["recommended_action"], str) and len(row["recommended_action"].strip()) > 10:
        score += 0.25
    if isinstance(row["confidence_level"], str) and row["confidence_level"] in ["high", "medium"]:
        score += 0.20
    if isinstance(row["ai_explanation"], str) and "Similar historical signals" in row["ai_explanation"]:
        score += 0.15
    if isinstance(row["ai_explanation"], str) and "Recommended action" in row["ai_explanation"]:
        score += 0.15

    return round(score, 2)

ai_explanations_df["explanation_score"] = ai_explanations_df.apply(score_explanation, axis=1)

avg_explanation_score = round(ai_explanations_df["explanation_score"].mean(), 2)
retrieval_avg_score = round(float(retrieved_results_df["score"].mean()), 2) if len(retrieved_results_df) > 0 else 0.0

system_feedback = []

if avg_explanation_score < 0.80:
    system_feedback.append("Explanation quality below target")
if retrieval_avg_score < 0.72:
    system_feedback.append("Retrieval relevance below target")
if len(system_feedback) == 0:
    system_feedback.append("System performing within expected range")

adaptive_rules_df = updated_rules_df.copy()

if retrieval_avg_score < 0.72:
    row_idx = adaptive_rules_df[adaptive_rules_df["rule_name"] == "retrieval_min_score"].index
    if len(row_idx) > 0:
        adaptive_rules_df.loc[row_idx, "rule_value"] = round(retrieval_avg_score + 0.05, 2)

if avg_explanation_score < 0.80:
    row_idx = adaptive_rules_df[adaptive_rules_df["rule_name"] == "high_usage_spike"].index
    if len(row_idx) > 0:
        current_value = float(adaptive_rules_df.loc[row_idx, "rule_value"].iloc[0])
        adaptive_rules_df.loc[row_idx, "rule_value"] = round(current_value + 0.03, 2)

evaluation_summary = pd.DataFrame({
    "metric": [
        "average_explanation_score",
        "average_retrieval_score",
        "feedback_message_count"
    ],
    "value": [
        avg_explanation_score,
        retrieval_avg_score,
        len(system_feedback)
    ]
})

feedback_messages_df = pd.DataFrame({
    "feedback_message": system_feedback
})

ai_explanations_df.to_csv(OUTPUT_DIR / "evaluated_ai_explanations.csv", index=False)
adaptive_rules_df.to_csv(OUTPUT_DIR / "adaptive_system_rules.csv", index=False)
evaluation_summary.to_csv(OUTPUT_DIR / "evaluation_summary.csv", index=False)
feedback_messages_df.to_csv(OUTPUT_DIR / "system_feedback_messages.csv", index=False)

print("Evaluation and self-improvement loop completed successfully")
print("\nEvaluation summary:")
print(evaluation_summary)
print("\nSystem feedback:")
print(feedback_messages_df)
print("\nAdaptive rules:")
print(adaptive_rules_df)

Evaluation and self-improvement loop completed successfully

Evaluation summary:
                      metric  value
0  average_explanation_score   1.00
1    average_retrieval_score   0.74
2     feedback_message_count   1.00

System feedback:
                          feedback_message
0  System performing within expected range

Adaptive rules:
             rule_name  rule_value  status
0     high_usage_spike        1.60  active
1     low_voltage_drop      108.00  active
2  retrieval_min_score        0.72  active


In [21]:
airflow_databricks_dag = r'''
from airflow import DAG
from airflow.providers.standard.operators.empty import EmptyOperator
from airflow.providers.databricks.operators.databricks import DatabricksRunNowOperator
from datetime import datetime

default_args = {
    "owner": "ashok",
    "start_date": datetime(2025, 1, 1),
    "retries": 1
}

with DAG(
    dag_id="autonomous_ai_data_platform_trigger",
    default_args=default_args,
    schedule="@daily",
    catchup=False,
    tags=["airflow", "databricks", "ai", "utility", "autonomous"]
) as dag:

    start_task = EmptyOperator(task_id="start_pipeline")

    trigger_databricks_job = DatabricksRunNowOperator(
        task_id="trigger_databricks_job",
        databricks_conn_id="databricks_default",
        job_id=123456
    )

    end_task = EmptyOperator(task_id="end_pipeline")

    start_task >> trigger_databricks_job >> end_task
'''

dag_file = PROJECT_DIR / "autonomous_ai_data_platform_airflow_databricks.py"
dag_file.write_text(airflow_databricks_dag, encoding="utf-8")

print("Airflow to Databricks DAG file created successfully")
print("File path:", dag_file.resolve())

Airflow to Databricks DAG file created successfully
File path: /content/Autonomous-AI-Data-Platform/autonomous_ai_data_platform_airflow_databricks.py


In [22]:
import zipfile

print("Airflow DAG preview:\n")
print(dag_file.read_text(encoding="utf-8"))

zip_path = Path("Autonomous-AI-Data-Platform-final.zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for file_path in PROJECT_DIR.rglob("*"):
        if file_path.is_file():
            zf.write(file_path, arcname=file_path.relative_to(PROJECT_DIR))

print("\nProject packaged successfully")
print("ZIP path:", zip_path.resolve())

Airflow DAG preview:


from airflow import DAG
from airflow.providers.standard.operators.empty import EmptyOperator
from airflow.providers.databricks.operators.databricks import DatabricksRunNowOperator
from datetime import datetime

default_args = {
    "owner": "ashok",
    "start_date": datetime(2025, 1, 1),
    "retries": 1
}

with DAG(
    dag_id="autonomous_ai_data_platform_trigger",
    default_args=default_args,
    schedule="@daily",
    catchup=False,
    tags=["airflow", "databricks", "ai", "utility", "autonomous"]
) as dag:

    start_task = EmptyOperator(task_id="start_pipeline")

    trigger_databricks_job = DatabricksRunNowOperator(
        task_id="trigger_databricks_job",
        databricks_conn_id="databricks_default",
        job_id=123456
    )

    end_task = EmptyOperator(task_id="end_pipeline")

    start_task >> trigger_databricks_job >> end_task


Project packaged successfully
ZIP path: /content/Autonomous-AI-Data-Platform-final.zip
